# 7.2. Import final manually cleaned comm groups

Imports manually cleaned comm groups entries. Creates final dataset of cleaned labgroupid x comm_group.

In [1]:
# Set-up
import pandas as pd
import numpy as np
import sys
from pathlib import Path
CODE_ROOT = Path.cwd().parents[0]
sys.path.append(str(CODE_ROOT))
import config

In [2]:
# Load data
labs = pd.read_csv(config.PROCESSED_DATA / "individual_processed_1.csv", keep_default_na=False, na_values=[""])

cleaned_comm_groups = pd.read_excel(
    config.CLEANING_WORKBOOKS / "comm_group_final.xlsx",
    keep_default_na=False,  # Keep "None" as a string, not NaN
    na_values=[""] # Only treat empty strings as NaN
)

In [3]:
# Keep only if "checked" == "Y"
cleaned_comm_groups = cleaned_comm_groups[cleaned_comm_groups["checked"] == "Y"]

# Keep only labgroupid and cleaned_value columns
cleaned_comm_groups = cleaned_comm_groups[["labgroupid", "cleaned_value"]]

In [4]:
# For all entries with several groups reported (separated by comma), split into separate rows
cleaned_comm_groups_long = cleaned_comm_groups.assign(
    group=cleaned_comm_groups["cleaned_value"].str.split(" / ")).explode("group").reset_index(drop=True)

In [5]:
# Drop all rows where "group" is missing
cleaned_comm_groups_long = cleaned_comm_groups_long.dropna(subset=["group"]).reset_index(drop=True)

In [6]:
# Create variable "sample_group" which is "group" (as an int) if "group" is a labgroupid in our sample
labgroupids = set(labs["labgroupid"])
group_numeric = pd.to_numeric(cleaned_comm_groups_long["group"], errors="coerce")
cleaned_comm_groups_long["sample_group"] = group_numeric.where(group_numeric.isin(labgroupids)).astype("Int64")

In [8]:
# Save cleaned comm groups to CSV
cleaned_comm_groups_long = cleaned_comm_groups_long[["labgroupid", "group", "sample_group"]]
cleaned_comm_groups_long.to_csv(config.CLEAN_DATA / "comm_groups_cleaned.csv", index=False)